In [1]:
import argparse
import numpy as np
import torch
import torch.nn.functional as F
import pickle
import platform
import os

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# !git clone https://github.com/vita-epfl/DLAV-2024.git
# path = os.getcwd() + '/DLAV-2024/homeworks/hw2/test_batch'
from pathlib import Path
path = Path.cwd() / 'test_batch'

In [4]:
# Write the location of the saved weight relative to this notebook. Assume that they are in the same directory
### Path to Model Weights 
softmax_weights = 'softmax_weights.pkl'
pytorch_weights = 'linearClassifier_pytorch.ckpt'

**TODO:** Copy your code from the Softmax Notebook to their corresponding function

In [5]:
def softmax_loss_vectorized(W, X, y):
    """
  Softmax loss function, vectorized version.
  Inputs and outputs are the same as softmax_loss_naive.
  """
    # Initialize the loss and gradient to zero.
    loss = 0.0
    dW = np.zeros_like(W)

    # Compute the softmax loss and its gradient using no explicit loops:
    N = X.shape[0]

    z = X.dot(W)  # N x C
    # Shift each row of scores by their max, for numerical stability
    z_shifted = z - np.max(z, axis=1, keepdims=True)  # N x C

    exp_z = np.exp(z_shifted)  # N x C - each row's max is 1
    sum_exp_z = np.sum(exp_z, axis=1, keepdims=True)  # N x 1
    probs = exp_z / sum_exp_z  # N x C

    true_class_probs = probs[np.arange(N), y]  # N x 1
    loss = -np.sum(np.log(true_class_probs)) / N  # Averaged over the batch

    # Gradient calculation
    dprobs = probs.copy()  # N x C
    dprobs[np.arange(N), y] -= 1  # Subtract 1 for the true class probabilities (now rows sum to 0)
    dprobs /= N  # Average over the batch
    dW = X.T.dot(dprobs)  # D x C
    
    return loss, dW

class LinearClassifier(object):

    def __init__(self):
        self.W = None


    def train(self, X, y, learning_rate=1e-3, num_iters=30000,
                batch_size=200, verbose=False):
        """
        Train this linear classifier using stochastic gradient descent.

        Inputs:
        - X: A numpy array of shape (N, D) containing training data; there are N
          training samples each of dimension D.
        - y: A numpy array of shape (N,) containing training labels; y[i] = c
          means that X[i] has label 0 <= c < C for C classes.
        - learning_rate: (float) learning rate for optimization.
        - num_iters: (integer) number of steps to take when optimizing
        - batch_size: (integer) number of training examples to use at each step.
        - verbose: (boolean) If true, print progress during optimization.

        Outputs:
        A list containing the value of the loss function at each training iteration.
        """
        
        num_train, dim = X.shape
        num_classes = np.max(y) + 1 # assume y takes values 0...K-1 where K is number of classes
        
        if self.W is None:
            # lazily initialize W
            self.W = 0.001 * np.random.randn(dim, num_classes)

        # Run stochastic gradient descent to optimize W
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None

            batch_idx = np.random.choice(num_train, batch_size, replace=True)
            X_batch = X[batch_idx]
            y_batch = y[batch_idx]

            # evaluate loss and gradient
            loss, grad = self.loss(X_batch, y_batch)
            loss_history.append(loss)

            # perform parameter update
            self.W -= learning_rate * grad

            if verbose and it % 100 == 0:
                print('iteration %d / %d: loss %f' % (it, num_iters, loss))


        return loss_history
    


    def predict(self, X):
        """
        Use the trained weights of this linear classifier to predict labels for
        data points.

        Inputs:
        - X: A numpy array of shape (N, D) containing training data; there are N
          training samples each of dimension D.

        Returns:
        - y_pred: Predicted labels for the data in X. y_pred is a 1-dimensional
          array of length N, and each element is an integer giving the predicted
          class.
        """

        scores = X.dot(self.W)
        y_pred = np.argmax(scores, axis=1)
        return y_pred

    def loss(self, X_batch, y_batch):
        """
        Compute the loss function and its derivative. 
        Subclasses will override this.

        Inputs:
        - X_batch: A numpy array of shape (N, D) containing a minibatch of N
          data points; each point has dimension D.
        - y_batch: A numpy array of shape (N,) containing labels for the minibatch.


        Returns: A tuple containing:
        - loss as a single float
        - gradient with respect to self.W; an array of the same shape as W
        
         e = y_batch - np.dot(X_batch, self.W) 
        
        loss = np.dot(e.T, e)
        grad = -np.dot(x_batch.T,e) / x_batch.shape[0]
  
        return loss, grad

        """

        pass
        


class Softmax(LinearClassifier):
    """ A subclass that uses the Softmax + Cross-entropy loss function """

    def loss(self, X_batch, y_batch):
        return softmax_loss_vectorized(self.W, X_batch, y_batch)

**TODO:** Copy the model you created from the Pytorch Notebook

In [6]:
class Net(torch.nn.Module):
    def __init__(self, n_feature=3072, n_hidden=512, n_output=10):
        super(Net, self).__init__()

        self.fc1 = torch.nn.Linear(n_feature, n_hidden)  # 3072 -> 512
        self.bn1 = torch.nn.BatchNorm1d(n_hidden)
        
        self.fc2 = torch.nn.Linear(n_hidden, n_hidden // 2)  # 512 -> 256
        self.bn2 = torch.nn.BatchNorm1d(n_hidden // 2)
        
        self.fc3 = torch.nn.Linear(n_hidden // 2, n_hidden // 4)  # 256 -> 128
        self.bn3 = torch.nn.BatchNorm1d(n_hidden // 4)
        
        self.fc4 = torch.nn.Linear(n_hidden // 4, n_output)  # 128 -> 10

    def forward(self, x):
        # Flatten the image
        x = x.view(x.size(0), -1)

        # Forward pass with BatchNorm and ReLU activations
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)  # No activation on the final layer, CrossEntropyLoss handles softmax internally

        return x

**TODO**: Follow the instructions in each of the following methods. **Note that these methods should return a 1-D array of size N where N is the number of data samples. The values should be the predicted classes [0,...,9].**



In [7]:
def predict_usingPytorch(X):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = Net(n_feature=3072, n_hidden=512, n_output=10).to(device)
    
    # Load weights
    try:
        model.load_state_dict(torch.load(pytorch_weights, map_location=device))
    except FileNotFoundError:
        print(f"Warning: {pytorch_weights} not found. Ensure model is trained and saved.")
        return np.zeros(X.shape[0])
        
    model.eval()
    with torch.no_grad():
        X = X.to(device)
        outputs = model(X)
        _, y_pred = torch.max(outputs.data, 1)
        
    return y_pred.cpu().numpy()

def predict_usingSoftmax(X):
    try:
        with open(softmax_weights, 'rb') as f:
            W = pickle.load(f)
    except FileNotFoundError:
        print(f"Warning: {softmax_weights} not found. Ensure Softmax was saved.")
        return np.zeros(X.shape[0])
            
    model = Softmax()
    model.W = W.copy()
    y_pred = model.predict(X)
    return y_pred

This method loads the test dataset to evaluate the model.

In [8]:
## Read DATA
def load_pickle(f):
    version = platform.python_version_tuple()
    if version[0] == '2':
        return  pickle.load(f)
    elif version[0] == '3':
        return  pickle.load(f, encoding='latin1')
    raise ValueError("invalid python version: {}".format(version))

def load_CIFAR_batch(filename):
  """ load single batch of cifar """
  with open(filename, 'rb') as f:
    datadict = load_pickle(f)
    X = datadict['data']
    Y = datadict['labels']
    X = X.reshape(10000, 3, 32, 32).transpose(0,2,3,1).astype("float")
    Y = np.array(Y)
    return X, Y
test_filename = path
X,Y = load_CIFAR_batch(test_filename)

This code snippet prepares the data for the different models. If you modify data manipulation in your notebooks, make sure to include them here. 

In [9]:
## Data Manipulation

mean = np.array([0.4914, 0.4822, 0.4465])
std = np.array([0.2023, 0.1994, 0.2010])
X = np.divide(np.subtract( X/255 , mean[np.newaxis,np.newaxis,:]), std[np.newaxis,np.newaxis,:])

X_pytorch = torch.Tensor(np.moveaxis(X,-1,1))
X_softmax = np.reshape(X, (X.shape[0], -1))
X_softmax = np.hstack([X_softmax, np.ones((X_softmax.shape[0], 1))])


Runs evaluation on the Pytorch and softmax model. **Be careful that *prediction_pytorch* and *prediction_softmax* are 1-D array of size N where N is the number of data samples. The values should be the predicted class [0,...,9]**

---



In [10]:
## Run Prediction
prediction_pytorch = predict_usingPytorch(X_pytorch)
prediction_softmax = predict_usingSoftmax(X_softmax)

## Run Evaluation
acc_softmax = sum(prediction_softmax == Y)/len(X)
acc_pytorch = sum(prediction_pytorch == Y)/len(X)
print("Softmax= %f ... Pytorch= %f"%(acc_softmax, acc_pytorch))

Softmax= 0.401300 ... Pytorch= 0.592700
